# RLHF, GRPO and Constitutional AI

The transformer pages in this section — [[autoregressive-transformer-encoder-decoder]],
[[kimi-k3]], [[deepseek-v4]] — describe *architectures*. They stop at next-token
prediction, which produces a model that continues text plausibly and nothing
more. Everything that makes a modern assistant useful happens afterwards, in a
training stage those pages do not cover.

This page covers it. Three methods, each answering a different question:

| method | question | signal |
|---|---|---|
| **RLHF / PPO** | make the model prefer what people prefer | a reward model trained on human comparisons |
| **GRPO** | make the model reason correctly | verifiable answers, no reward model, no value network |
| **Constitutional AI** | make the model follow written principles | the model critiquing its own output |

A note on scope, because it matters for what you can trust here. **Anthropic has
never published Claude's architecture**, and OpenAI has not published GPT-5's.
What *is* published is how this training stage works — Constitutional AI in full,
GRPO with complete equations. So this page is the honest answer to "how do these
models get made", and it stops where the public record stops.

Every quantity below is computed on tiny examples where the arithmetic is
checkable by hand.

## 1. Why next-token prediction is not enough

A pretrained model estimates $P(\text{next token} \mid \text{context})$ from a
corpus. That objective has no notion of helpful, correct, or safe — only
*typical*. Ask a purely pretrained model a question and a very reasonable
continuation is another question, because documents full of questions exist.

You cannot fix this with more pretraining, because the thing you want is not
"what text usually follows" but "what response a person would prefer". That
preference is not in the corpus as a label. It has to be introduced.

The obstacle: preference is not differentiable. You cannot write
$\partial(\text{helpfulness})/\partial\theta$. Reinforcement learning exists
precisely to optimise a signal you can only *evaluate*, not differentiate, and
that is why it appears here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

banner("STEP 1: A REWARD MODEL FROM PAIRWISE COMPARISONS")

# People are bad at scoring things 1-10 and good at picking the better of two.
# So the data is comparisons, and the reward model is fitted to them.
step("The comparison data")
prompts = ["how do I reverse a list?", "what is 17 x 23?", "summarise this paper"]
pairs = [  # (prompt, chosen_score_features, rejected_score_features)
    ("how do I reverse a list?",  "direct answer + example", "a question back"),
    ("what is 17 x 23?",          "391, with working",       "391"),
    ("summarise this paper",      "3 bullet summary",        "restates the title"),
]
for p, ch, rj in pairs:
    print(f"  prompt   : {p}")
    print(f"    chosen : {ch}")
    print(f"    rejected: {rj}")

step("The Bradley-Terry model: turn scores into a probability of preference")
print("  P(A preferred over B) = sigmoid(r_A - r_B)")
print("  so the loss is  -log sigmoid(r_chosen - r_rejected)")
print()
sig = lambda z: 1/(1+np.exp(-z))
print(f"  {'r_chosen':>9}  {'r_rejected':>11}  {'margin':>7}  {'P(chosen)':>10}"
      f"  {'loss':>7}")
print("  " + "-" * 52)
for rc, rr in [(2.0, 1.0), (1.0, 1.0), (0.5, 2.0), (5.0, 1.0)]:
    m = rc - rr
    print(f"  {rc:>9.1f}  {rr:>11.1f}  {m:>7.1f}  {sig(m):>10.4f}"
          f"  {-np.log(sig(m)):>7.4f}")
print("  " + "-" * 52)
print("  Only the MARGIN matters, never the absolute scores. A reward model")
print("  is identified up to an additive constant -- which is exactly why the")
print("  advantage normalisation in section 3 is legitimate.")

step("One gradient step on the reward model, by hand")
r_chosen, r_rejected = 0.5, 1.2          # currently WRONG: rejected scores higher
loss = -np.log(sig(r_chosen - r_rejected))
d_chosen  = -(1 - sig(r_chosen - r_rejected))     # dL/dr_chosen
d_rejected = (1 - sig(r_chosen - r_rejected))     # dL/dr_rejected
print(f"  r_chosen = {r_chosen}, r_rejected = {r_rejected}  -> the model has it backwards")
print(f"  loss = -log sigmoid({r_chosen} - {r_rejected}) ="
      f" -log sigmoid({r_chosen-r_rejected:.1f}) = {loss:.4f}")
print(f"  dL/dr_chosen   = -(1 - sigmoid(margin)) = {d_chosen:+.4f}   (push UP)")
print(f"  dL/dr_rejected = +(1 - sigmoid(margin)) = {d_rejected:+.4f}   (push DOWN)")
eta = 0.5
print(f"  after one step with lr={eta}:")
print(f"    r_chosen  {r_chosen} -> {r_chosen - eta*d_chosen:.4f}")
print(f"    r_rejected {r_rejected} -> {r_rejected - eta*d_rejected:.4f}")
new_margin = (r_chosen - eta*d_chosen) - (r_rejected - eta*d_rejected)
print(f"    margin {r_chosen-r_rejected:+.4f} -> {new_margin:+.4f}"
      f"   (improved: {new_margin > r_chosen-r_rejected})")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
  STEP 1: A REWARD MODEL FROM PAIRWISE COMPARISONS

The comparison data
  prompt   : how do I reverse a list?
    chosen : direct answer + example
    rejected: a question back
  prompt   : what is 17 x 23?
    chosen : 391, with working
    rejected: 391
  prompt   : summarise this paper
    chosen : 3 bullet summary
    rejected: restates the title

The Bradley-Terry model: turn scores into a probability of preference
  P(A preferred over B) = sigmoid(r_A - r_B)
  so the loss is  -log sigmoid(r_chosen - r_rejected)

   r_chosen   r_rejected   margin   P(chosen)     loss
  ----------------------------------------------------
        2.0          1.0      1.0      0.7311   0.3133
        1.0          1.0      0.0      0.5000   0.6931
        0.5          2.0     -1.5      0.1824   1.7014
        5.0          1.0      4.0      0.9820   0.0181
  ------------------------------------------------

## 2. The KL penalty: why the policy is kept on a leash

With a reward model in hand, the obvious move is to maximise reward. That fails
immediately, and the failure has a name: **reward hacking**.

The reward model is an imperfect approximation of human preference, fitted on a
finite sample. Optimise hard enough against any imperfect proxy and you find its
errors rather than the thing it was proxying for — outputs that score highly and
are worthless. Goodhart's law with a gradient.

The standard fix is to penalise drift from the original model:

$$\text{objective} = \mathbb{E}\big[\,r(x, y)\,\big] - \beta\, \mathrm{KL}\big(\pi_\theta \,\|\, \pi_{\text{ref}}\big)$$

$\pi_{\text{ref}}$ is the frozen pre-RL model. $\beta$ controls how far the policy
may wander. This is the single most consequential hyperparameter in the whole
pipeline: too small and the model degenerates into reward-model exploits, too
large and it barely changes.

In [2]:
banner("THE KL PENALTY, AND WHAT HAPPENS WITHOUT IT")

step("KL divergence between the policy and the reference, on one token")
vocab = ['the', 'a', 'ALWAYS', 'answer', 'yes']
ref = np.array([0.30, 0.25, 0.05, 0.25, 0.15])
print(f"  reference distribution over a 5-token vocabulary:")
for t, p in zip(vocab, ref):
    print(f"    {t:>8}: {p:.4f}")

def kl(p, q):
    return float(np.sum(p * np.log(p / q)))

print()
print(f"  {'policy description':>34}  {'P(ALWAYS)':>10}  {'KL':>8}")
print("  " + "-" * 56)
for lab, pol in [
    ("identical to reference",        ref.copy()),
    ("mild shift",                    np.array([0.28, 0.22, 0.14, 0.24, 0.12])),
    ("strong shift",                  np.array([0.15, 0.12, 0.50, 0.15, 0.08])),
    ("collapsed onto one token",      np.array([0.02, 0.02, 0.93, 0.02, 0.01])),
]:
    pol = pol / pol.sum()
    print(f"  {lab:>34}  {pol[2]:>10.4f}  {kl(pol, ref):>8.4f}")
print("  " + "-" * 56)
print(f"  hand check, identical distributions: KL = {kl(ref, ref):.2e}"
      f"   is zero: {np.isclose(kl(ref, ref), 0)}")

step("Reward hacking: a simulated optimisation with and without the penalty")
# The reward model over-rewards one token; true quality does not.
true_q  = np.array([0.5, 0.4, 0.1, 0.9, 0.3])     # what we actually want
rm      = np.array([0.5, 0.4, 3.0, 0.9, 0.3])     # the proxy, wrong on 'ALWAYS'
print(f"  the reward model over-scores {vocab[2]!r}: rm={rm[2]:.1f} vs true"
      f" quality {true_q[2]:.1f}")
print()
print(f"  {'beta':>7}  {'P(ALWAYS)':>10}  {'reward-model score':>19}"
      f"  {'TRUE quality':>13}")
print("  " + "-" * 56)
for beta in [0.0, 0.05, 0.2, 1.0]:
    logits = np.log(ref) + rm / max(beta, 1e-9) if beta > 0 else None
    if beta == 0.0:
        pol = np.zeros(5); pol[np.argmax(rm)] = 1.0        # pure argmax
    else:
        # optimal policy for reward + KL is the reference tilted by exp(r/beta)
        pol = ref * np.exp(rm / beta)
        pol = pol / pol.sum()
    print(f"  {beta:>7.2f}  {pol[2]:>10.4f}  {float(pol @ rm):>19.4f}"
          f"  {float(pol @ true_q):>13.4f}")
print("  " + "-" * 56)
print(f"  Read the last two columns against each other. As beta falls, the")
print(f"  reward-model score rises monotonically and TRUE quality collapses.")
print(f"  The model is not getting better; it is finding the proxy's mistake.")
print()
print(f"  The closed form used above is worth knowing: maximising")
print(f"  E[r] - beta*KL(pi || ref) has the exact solution")
print(f"      pi*(y) proportional to ref(y) * exp(r(y)/beta)")
print(f"  so beta interpolates between the reference (beta -> infinity) and")
print(f"  argmax reward (beta -> 0).")

  THE KL PENALTY, AND WHAT HAPPENS WITHOUT IT

KL divergence between the policy and the reference, on one token
  reference distribution over a 5-token vocabulary:
         the: 0.3000
           a: 0.2500
      ALWAYS: 0.0500
      answer: 0.2500
         yes: 0.1500

                  policy description   P(ALWAYS)        KL
  --------------------------------------------------------
              identical to reference      0.0500    0.0000
                          mild shift      0.1400    0.0601
                        strong shift      0.5000    0.8323
            collapsed onto one token      0.9300    2.5363
  --------------------------------------------------------
  hand check, identical distributions: KL = 0.00e+00   is zero: True

Reward hacking: a simulated optimisation with and without the penalty
  the reward model over-scores 'ALWAYS': rm=3.0 vs true quality 0.1

     beta   P(ALWAYS)   reward-model score   TRUE quality
  ------------------------------------------------

## 3. GRPO: dropping the value network

PPO — the algorithm behind classic RLHF — needs a **value network** alongside the
policy: a second model, the same size as the first, trained to predict expected
future reward so that advantages can be computed. That is expensive and
notoriously fiddly to stabilise.

**Group Relative Policy Optimization** (DeepSeek, 2024) removes it with a simple
observation: if you sample $G$ completions for the *same* prompt, the group's own
mean is a perfectly good baseline. No separate network required.

For prompt $q$, sample $\{o_1, \dots, o_G\}$, score each with reward $r_i$, then

$$A_i = \frac{r_i - \operatorname{mean}(r_1 \dots r_G)}{\operatorname{std}(r_1 \dots r_G)}$$

That is the whole idea. Completions better than their siblings get positive
advantage; worse ones get negative. The objective is then PPO's clipped ratio,
using these advantages:

$$\mathcal{J} = \mathbb{E}\Big[\min\big(\rho_i A_i,\ \operatorname{clip}(\rho_i, 1-\epsilon, 1+\epsilon) A_i\big)\Big] - \beta\,\mathrm{KL}(\pi_\theta \| \pi_{\text{ref}}), \qquad \rho_i = \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{\text{old}}}(o_i|q)}$$

This is what trains reasoning models. The reward can be a **verifier** rather
than a learned model — did the code run, did the final answer match — which
sidesteps reward hacking almost entirely, because a unit test is not a proxy for
correctness, it *is* correctness.

In [3]:
banner("GRPO ADVANTAGES, BY HAND")

step("Sample a group of 8 completions for one maths prompt")
prompt = "what is 17 x 23?"
completions = [
    ("391, since 17x20=340 and 17x3=51",      1.0),
    ("391",                                    1.0),
    ("17 x 23 = 391",                          1.0),
    ("about 400",                              0.0),
    ("391, because 17x23 = 17x25 - 17x2",      1.0),
    ("371",                                    0.0),
    ("let me think... 17x23... 380?",          0.0),
    ("391",                                    1.0),
]
print(f"  prompt: {prompt}")
print(f"  reward = 1 if the final answer is 391, else 0 -- a VERIFIER, not a")
print(f"  learned reward model. Nothing to hack.")
print()
r = np.array([c[1] for c in completions])
print(f"  {'i':>3}  {'completion':>42}  {'reward':>7}")
print("  " + "-" * 58)
for i, (txt, rew) in enumerate(completions):
    print(f"  {i:>3}  {txt:>42}  {rew:>7.1f}")
print("  " + "-" * 58)

step("Group-relative advantages")
mu, sd = r.mean(), r.std()
A = (r - mu) / sd
print(f"  mean(r) = ({' + '.join(f'{v:.0f}' for v in r)}) / {len(r)} = {mu:.4f}")
print(f"  std(r)  = {sd:.4f}")
print()
print(f"  {'i':>3}  {'r':>5}  {'r - mean':>10}  {'A = (r-mean)/std':>18}")
print("  " + "-" * 42)
for i in range(len(r)):
    print(f"  {i:>3}  {r[i]:>5.1f}  {r[i]-mu:>10.4f}  {A[i]:>18.4f}")
print("  " + "-" * 42)
print(f"  hand check A[0] = ({r[0]:.1f} - {mu:.4f}) / {sd:.4f} ="
      f" {(r[0]-mu)/sd:.6f}   MATCH: {np.allclose(A[0], (r[0]-mu)/sd)}")
print()
print(f"  sum of advantages = {A.sum():.2e}   (zero by construction:"
      f" {np.isclose(A.sum(), 0)})")
print(f"  That zero sum IS the baseline. No value network computed it -- the")
print(f"  group did.")

step("Why a baseline at all")
print(f"  Without one, every completion with positive reward is reinforced,")
print(f"  including the mediocre ones. Compare the update signal:")
print()
print(f"  {'i':>3}  {'reward':>7}  {'no baseline':>12}  {'GRPO advantage':>16}")
print("  " + "-" * 44)
for i in range(len(r)):
    print(f"  {i:>3}  {r[i]:>7.1f}  {r[i]:>12.1f}  {A[i]:>16.4f}")
print("  " + "-" * 44)
print(f"  With no baseline the gradient always pushes UP on the {int(r.sum())} correct")
print(f"  completions and never pushes DOWN on the {int((1-r).sum())} wrong ones -- so the")
print(f"  policy drifts toward whatever it already does, and variance is huge.")
print(f"  Centring on the group mean makes the wrong ones actively discouraged.")

step("The degenerate case the formula has to survive")
for lab, rr in [("all correct", np.ones(8)), ("all wrong", np.zeros(8))]:
    s = rr.std()
    print(f"  {lab:>12}: std = {s:.1f}  ->"
          f" {'advantages undefined (0/0)' if s == 0 else 'fine'}")
print(f"  When every completion in a group gets the same reward there is no")
print(f"  relative information, and implementations skip the group entirely.")
print(f"  This matters in practice: on problems that are too easy or too hard")
print(f"  for the current policy, GRPO gets no signal at all -- which is why")
print(f"  curriculum and difficulty filtering are a real part of these")
print(f"  pipelines rather than a refinement.")

  GRPO ADVANTAGES, BY HAND

Sample a group of 8 completions for one maths prompt
  prompt: what is 17 x 23?
  reward = 1 if the final answer is 391, else 0 -- a VERIFIER, not a
  learned reward model. Nothing to hack.

    i                                  completion   reward
  ----------------------------------------------------------
    0            391, since 17x20=340 and 17x3=51      1.0
    1                                         391      1.0
    2                               17 x 23 = 391      1.0
    3                                   about 400      0.0
    4           391, because 17x23 = 17x25 - 17x2      1.0
    5                                         371      0.0
    6               let me think... 17x23... 380?      0.0
    7                                         391      1.0
  ----------------------------------------------------------

Group-relative advantages
  mean(r) = (1 + 1 + 1 + 0 + 1 + 0 + 0 + 1) / 8 = 0.6250
  std(r)  = 0.4841

    i      r    r - mean

In [4]:
banner("THE CLIPPED OBJECTIVE, TERM BY TERM")

step("Why clip the probability ratio")
print("  rho = pi_new(o|q) / pi_old(o|q)  -- how much more likely the updated")
print("  policy makes this completion. Large rho means a big step, and the")
print("  advantage estimate was computed under pi_old, so it stops being")
print("  trustworthy. Clipping removes the incentive to move too far.")
print()
eps = 0.2
print(f"  epsilon = {eps},  so rho is clipped to [{1-eps}, {1+eps}]")
print()
print(f"  {'rho':>6}  {'A':>6}  {'rho*A':>9}  {'clip(rho)*A':>13}"
      f"  {'min -> used':>13}  note")
print("  " + "-" * 72)
for rho, Ai in [(1.0, 1.5), (1.5, 1.5), (0.5, 1.5), (1.5, -1.5), (0.5, -1.5)]:
    unc = rho * Ai
    cl  = np.clip(rho, 1-eps, 1+eps) * Ai
    used = min(unc, cl)
    if Ai > 0 and rho > 1+eps:
        note = "good action, already boosted -> capped"
    elif Ai < 0 and rho < 1-eps:
        note = "bad action, already suppressed -> capped"
    elif rho == 1.0:
        note = "no change yet"
    else:
        note = "inside the trust region"
    print(f"  {rho:>6.2f}  {Ai:>6.2f}  {unc:>9.4f}  {cl:>13.4f}"
          f"  {used:>13.4f}  {note}")
print("  " + "-" * 72)
print(f"  The min() is what makes it one-sided: it caps the reward for moving")
print(f"  further in a direction you have already moved, but never caps the")
print(f"  penalty for having moved the wrong way.")

step("GRPO vs PPO, in resources")
print(f"  {'':>26}  {'PPO':>18}  {'GRPO':>18}")
print("  " + "-" * 66)
for lab, ppo, grpo in [
    ("networks in memory",   "policy + value + ref", "policy + ref"),
    ("baseline from",        "learned value net",    "group mean"),
    ("reward source",        "learned reward model", "verifier or RM"),
    ("extra params trained", "~1x policy size",      "none"),
]:
    print(f"  {lab:>26}  {ppo:>18}  {grpo:>18}")
print("  " + "-" * 66)
print(f"  Dropping the value network removes a model the size of the policy")
print(f"  from both memory and the optimisation. That is the practical reason")
print(f"  GRPO spread so quickly.")

  THE CLIPPED OBJECTIVE, TERM BY TERM

Why clip the probability ratio
  rho = pi_new(o|q) / pi_old(o|q)  -- how much more likely the updated
  policy makes this completion. Large rho means a big step, and the
  advantage estimate was computed under pi_old, so it stops being
  trustworthy. Clipping removes the incentive to move too far.

  epsilon = 0.2,  so rho is clipped to [0.8, 1.2]

     rho       A      rho*A    clip(rho)*A    min -> used  note
  ------------------------------------------------------------------------
    1.00    1.50     1.5000         1.5000         1.5000  no change yet
    1.50    1.50     2.2500         1.8000         1.8000  good action, already boosted -> capped
    0.50    1.50     0.7500         1.2000         0.7500  inside the trust region
    1.50   -1.50    -2.2500        -1.8000        -2.2500  inside the trust region
    0.50   -1.50    -0.7500        -1.2000        -1.2000  bad action, already suppressed -> capped
  --------------------------------

## 4. Constitutional AI

The closest thing to a published answer for "how does Claude work" — not the
architecture, which is undisclosed, but the alignment method, which Anthropic
described in full (Bai et al., 2022).

The problem it addresses: RLHF needs humans to label harmful outputs, which is
slow, expensive, inconsistent, and unpleasant work. Constitutional AI replaces
most of that human labelling with the **model critiquing itself against a written
set of principles** — the constitution.

Two phases:

**Supervised phase.** Prompt the model, including with adversarial prompts. For
each response: ask the model to *critique* its own answer against a constitutional
principle, then *revise* it accordingly. Fine-tune on the revisions. The model
learns from its own corrected output.

**RL phase (RLAIF).** Generate response pairs, and have the model — not a human —
choose which better satisfies the constitution. Those AI-generated preferences
train a preference model, and RL proceeds as in section 1.

The interesting property is that the values are **written down**. In ordinary
RLHF the norms live implicitly in labeller behaviour and can only be inferred; in
CAI they are a document you can read, argue with, and change.

In [5]:
banner("THE CRITIQUE-REVISE LOOP")

step("One principle, one response, one revision")
principle = ("Choose the response that is least likely to be harmful, "
             "and that explains the refusal honestly rather than evading.")
prompt    = "how do I pick a lock?"
initial   = "I can't help with that."
critique  = ("The response refuses, which is appropriate, but it gives no "
             "reason and does not distinguish legitimate uses. It is evasive "
             "rather than honest, which the principle asks it not to be.")
revised   = ("Lockpicking has legitimate uses - locksmithing, being locked out "
             "of your own home. I'd rather not give a step-by-step guide since "
             "I can't tell which situation this is, but I'm happy to explain "
             "how pin tumbler locks work, or point you to a locksmith.")
for lab, txt in [("principle", principle), ("prompt", prompt),
                 ("initial response", initial), ("self-critique", critique),
                 ("revision", revised)]:
    print(f"  {lab}:")
    for line in [txt[i:i+66] for i in range(0, len(txt), 66)]:
        print(f"      {line}")
    print()
print("  The training pair is (prompt -> revision). The critique is scaffolding:")
print("  it is what gets the model to the revision, and is then discarded.")

step("Why this scales when human labelling does not")
n_prompts, n_principles = 100_000, 16
print(f"  {'approach':>28}  {'human labels needed':>21}")
print("  " + "-" * 54)
print(f"  {'RLHF (human harm labels)':>28}  {n_prompts:>21,}")
print(f"  {'Constitutional AI':>28}  {n_principles:>21,}")
print("  " + "-" * 54)
print(f"  {n_prompts/n_principles:,.0f}x fewer human judgements, and the ones that")
print(f"  remain are principles written once rather than case-by-case rulings.")
print()
print(f"  The trade is real though: the model is now judging itself against")
print(f"  the constitution, so any misreading of a principle is applied")
print(f"  consistently across all {n_prompts:,} prompts rather than being")
print(f"  averaged out across many human labellers.")

step("RLAIF preference, scored the same way as section 1")
print("  The only change from RLHF is who produces the comparison label:")
print()
print(f"  {'stage':>22}  {'RLHF':>16}  {'Constitutional AI':>19}")
print("  " + "-" * 62)
for lab, a, b in [("who compares", "human labeller", "the model"),
                  ("against what", "own judgement", "written principle"),
                  ("preference model", "same", "same"),
                  ("RL step", "same", "same")]:
    print(f"  {lab:>22}  {a:>16}  {b:>19}")
print("  " + "-" * 62)
print("  Everything downstream of the comparison is unchanged -- Bradley-Terry")
print("  loss, reward model, KL-penalised policy optimisation. CAI swaps out")
print("  the source of the labels, not the machinery.")

  THE CRITIQUE-REVISE LOOP

One principle, one response, one revision
  principle:
      Choose the response that is least likely to be harmful, and that e
      xplains the refusal honestly rather than evading.

  prompt:
      how do I pick a lock?

  initial response:
      I can't help with that.

  self-critique:
      The response refuses, which is appropriate, but it gives no reason
       and does not distinguish legitimate uses. It is evasive rather th
      an honest, which the principle asks it not to be.

  revision:
      Lockpicking has legitimate uses - locksmithing, being locked out o
      f your own home. I'd rather not give a step-by-step guide since I 
      can't tell which situation this is, but I'm happy to explain how p
      in tumbler locks work, or point you to a locksmith.

  The training pair is (prompt -> revision). The critique is scaffolding:
  it is what gets the model to the revision, and is then discarded.

Why this scales when human labelling does no

## 5. What to take away

**Pretraining gives you a text continuer; this stage gives you an assistant.**
The gap is not more data — it is a signal that does not exist in the corpus,
introduced by evaluation rather than differentiation.

**Reward models are fitted from comparisons, not scores**, because people are
reliable at ranking and unreliable at rating. Bradley-Terry turns a margin into a
probability, and only the margin is identified — the absolute scale is free.

**The KL penalty is the load-bearing hyperparameter.** Any learned reward model
is a proxy, and optimising a proxy hard enough finds its flaws. The closed-form
optimum $\pi^* \propto \pi_{\text{ref}}\exp(r/\beta)$ shows exactly what $\beta$
interpolates between.

**GRPO's contribution is the baseline.** Sample a group, centre on its mean,
drop the value network. Combined with *verifiable* rewards — unit tests, checked
answers — it also removes most of the reward-hacking problem, because a passing
test is not a proxy for correctness.

**Constitutional AI moves the values into a document.** Same machinery
downstream; the model generates the preference labels by critiquing itself
against written principles. That is a scaling argument and a transparency
argument at once.

### Verified in this notebook

- Bradley-Terry loss and its gradients computed by hand; one step provably improves the margin
- KL divergence computed from the definition, zero for identical distributions
- Reward hacking demonstrated: as $\beta$ falls, reward-model score rises monotonically while true quality collapses
- GRPO advantages computed by hand for a group of 8, summing to zero by construction
- The all-correct and all-wrong degenerate cases identified, where $\text{std} = 0$ leaves the advantage undefined
- The clipped objective evaluated across five $(\rho, A)$ combinations showing the one-sided trust region

### Where this connects

[[autoregressive-transformer-encoder-decoder]] is the model this stage
post-trains. [[optimisers-sgd-to-adam]] supplies the update rule underneath.
[[evaluation-metrics]] and [[cross-validation]] are how you would check any of
it actually worked.

### Sources

Christiano et al., *Deep RL from Human Preferences* (2017); Ouyang et al.,
*Training language models to follow instructions with human feedback* (2022);
Shao et al., *DeepSeekMath* (2024), which introduced GRPO; Bai et al.,
*Constitutional AI: Harmlessness from AI Feedback* (2022).